# Unmodified TPCRP Algorithm
* This is the unmodified algorithm I made
* The Source of the paper is listed on the README.md - linking to the paper from KEATS
* An alternative link is present linking to the non-KEATS directed version of the same paper 

In [1]:
# Imports section - please run this before the algorithm 

import os
import numpy as np 


import torch 
import torch.nn as nn 
import torch.nn.functional as F 

import torchvision 
import torchvision.transforms as transforms 
from torch.utils.data import DataLoader, Dataset 

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

from typing import List, Tuple 

In [2]:
# [optional] - for reproducibility in results 
torch.manual_seed(0)
np.random.seed(0)

In [3]:
class ResNetEncd(nn.Module):
    def __init__(self, base="resnet18", proj_dim=128):
        super().__init__()

        backbone = getattr(torchvision.models, base)(pretrained=False)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        feat_dim = backbone.fc.in_features

        self.projection = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim)
        )

    def forward(self, x, return_projection: bool = True):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        if return_projection:
            proj = self.projection(feat)
            return feat, proj
        return feat 
    
    def represent(self, x):
        return self.forward(x, return_projection=False)

In [4]:
# NT-Xent Loss  - CONSTRASTIVE NTXent 
class ContrastiveNTXent(nn.Module):
    def __init__(self, temperature : float = 0.2):
        super().__init__()
        self.temperature = temperature 


    def forward(self, z_a: torch.Tensor, z_b : torch.Tensor) -> torch.Tensor:

        batch = z_a.size(0)
        z = torch.cat([z_a, z_b], dim =0 ) 
        z = F.normalize(z, dim=1)


        sim_matrix = torch.matmul(z, z.T) / self.temperature

        diag_mask = torch.eye(2 * batch, device= z.device).bool() 
        sim_matrix.masked_fill_(diag_mask, -9e15)

        labels = torch.arange(batch, device=z.device) # 1D-tensor
        labels = torch.cat([labels + batch, labels], dim=0)

        loss = F.cross_entropy(sim_matrix, labels)
        return loss 

In [5]:
# 2-Crop Transformation
class TC_Transform:
    def __init__(self):
        self.aug = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.2,1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.2470, 0.2435, 0.2616)),
        ])
    
    
    def __call__(self, img):
        return self.aug(img), self.aug(img)


In [6]:
def compute_typicality(features: np.ndarray, k: int = 20) -> np.ndarray:

    nbrs = NearestNeighbors(n_neighbors=min(k + 1, len(features)), algorithm='auto').fit(features)
    distances, _ = nbrs.kneighbors(features)  # distances: [N, k+1]
    distances = distances[:, 1:]
    avg_sq_dist = (distances ** 2).mean(axis=1)
    typicality = 1.0 / (avg_sq_dist + 1e-12)
    return typicality


def typiclust_initial_pool(
    features: np.ndarray,
    budget: int,
    k_typicality: int = 20,
    random_state: int = 0
) -> List[int]:
    
    n_samples = features.shape[0]
    n_clusters = budget

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(features)

    typicality = compute_typicality(features, k=k_typicality)

    selected_indices = []
    for c in range(n_clusters):
        cluster_idx = np.where(cluster_labels == c)[0]
        if len(cluster_idx) == 0:
            continue
        cluster_typ = typicality[cluster_idx]
        best_local = cluster_idx[np.argmax(cluster_typ)]
        selected_indices.append(int(best_local))

    return selected_indices


In [7]:
class TwoCropCIFAR_10(Dataset):
    def __init__(self, root: str, train:bool, transform):
        self.dataset = torchvision.datasets.CIFAR10(
            root = root,
            train= train,
            download = True,
            transform = None
        )
        self.transform = transform 

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        x1, x2 = self.transform(img)
        return x1, x2, idx 
    


def train_self_supervised(
    encoder : ResNetEncd,
    dataset: Dataset,
    device: torch.device,
    batch_size: int = 256,
    epochs: int = 200,
    lr: float = 1e-3
) -> ResNetEncd:
    loader = DataLoader(
        dataset,   # TO DO - ADD DATASET & BATCH_SIZE integration 
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        drop_last=True
    )


    encoder = encoder.to(device)
    loss_fn = ContrastiveNTXent(temperature=0.2).to(device)
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)

    encoder.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for x1, x2, _ in loader:
            x1= x1.to(device)
            x2 = x2.to(device)

            _, z1 = encoder(x1, return_projection=True)

            _, z2 = encoder(x2, return_projection=True)

            loss = loss_fn(z1,z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x1.size(0)

        epoch_loss = running_loss / len(loader.dataset)
        print(f"Epoch {epoch+1}/{epochs} - SSL LOSS : {epoch_loss:.4f}")

    return encoder 

In [8]:
# Extraction of Features
def extract_features(
        encoder : ResNetEncd,
        dataset : torchvision.datasets.CIFAR10,
        device : torch.device,
        batch_size : int = 256
) -> np.ndarray:
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers =4 
    )

    encoder.eval()
    all_features = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            features = encoder.represent(x) 
            all_features.append(features.cpu().numpy())

    features = np.concatenate(all_features, axis=0)
    return features 



# SSL = Self Supervised Learning 
def run_pipeline(
    data_root : str = "./data", # data folder for data root location 
    budget : int = 30,
    ssl_epochs : int = 200
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    # CIFAR-10 Two-Crop SSL
    ssl_transform = TC_Transform()
    ssl_dataset = TwoCropCIFAR_10(
        root= data_root,
        train=True,
        transform=ssl_transform
    )


    # Self-Supervised Encoder 
    encoder = ResNetEncd(base="resnet18", proj_dim=128)
    encoder = train_self_supervised(
        encoder = encoder,
        dataset=ssl_dataset,
        device=device,
        batch_size=256,
        epochs=ssl_epochs,
        lr=1e-3
    )

    # CIFAR -10 Feature Extraction 
    base_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),
            (0.2470, 0.2435, 0.2616)
        ),
    ])

    base_dataset = torchvision.datasets.CIFAR10(
        root=data_root,
        train=True,
        download=True,
        transform=base_transform
    )

    # Feat. Extraction
    features = extract_features(
        encoder=encoder,
        dataset=base_dataset,
        device=device,
        batch_size=256
    )


    # TypiClust Pool Select
    selected_indices = typiclust_initial_pool(
        features=features,
        budget=budget,
        k_typicality=20,
        random_state=0
    )

    os.makedirs("results", exist_ok=True)
    np.save("results/typiclust.npy", np.array(selected_indices))
    print(f"Saved { len(selected_indices)} to results/typiclust.npy")
    
    return selected_indices

In [9]:
if __name__ == '__main__':

    # The Epochs can be upped - will increasing training time 
    run_pipeline(ssl_epochs=10) # Run pipeline()

100.0%
/home/ariag/5CCSAMLF/second_coursework/env/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/home/ariag/5CCSAMLF/second_coursework/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ariag/5CCSAMLF/second_coursework/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1/10 - SSL LOSS : 4.7720
Epoch 2/10 - SSL LOSS : 4.1999
Epoch 3/10 - SSL LOSS : 3.9777
Epoch 4/10 - SSL LOSS : 3.8300
Epoch 5/10 - SSL LOSS : 3.7347
Epoch 6/10 - SSL LOSS : 3.6505
Epoch 7/10 - SSL LOSS : 3.5795
Epoch 8/10 - SSL LOSS : 3.5262
Epoch 9/10 - SSL LOSS : 3.4674
Epoch 10/10 - SSL LOSS : 3.4272
Saved 30 to results/typiclust.npy
